# Rebuild the meshwork `.h5` files and the subtype table

Panels S6b, S6c, S6d–h and S6j read `pcg_skel` **meshwork** exports: per-neuron files
carrying the level-2 skeleton, the mesh, and the neuron's synapses. Those files ship with
this repository (`../../data/meshwork_h5/`, 80 files + 8 Aδ-HTMR files), so the figure
notebooks run offline. This notebook only needs to be run to rebuild them from CAVE.

Each file is named by the seed coordinate used to look the neuron up, `x_y_z.h5`.

Synapses come from the CAVE **`sensory_neuron_synapses`** table — the manually curated set
— so `post_syn` (axoaxonic synapses onto the sensory axon) is the validated set rather
than raw predictions.


In [ ]:
import caveclient
import numpy as np
import pcg_skel

MATERIALIZATION = 669
RESOLUTION = [32, 32, 45]          # voxel size the seed coordinates are given in
SYNAPSE_TABLE = "sensory_neuron_synapses"

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION


In [ ]:
from pathlib import Path

OUT_DIR = Path("../../data/meshwork_h5")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def build(root_point, root_id=None):
    """Export one neuron as a meshwork .h5 named by its seed coordinate."""
    coord_str = "_".join(map(str, root_point))
    out = OUT_DIR / f"{coord_str}.h5"
    if out.exists():
        print(f"skip (exists): {out.name}")
        return

    if root_id is None:
        root_id = int(client.chunkedgraph.get_root_id(
            client.chunkedgraph.get_supervoxel_id(root_point, RESOLUTION)
        ))

    neuron = pcg_skel.pcg_meshwork(
        root_id=root_id,
        client=client,
        root_point=root_point,
        root_point_resolution=RESOLUTION,
        collapse_soma=False,
        synapses="all",
        synapse_table=SYNAPSE_TABLE,
        remove_self_synapse=True,
    )
    neuron.save_meshwork(out)
    print(f"saved {out.name}")


# build(root_point=[20373, 3883, 152])


## Subtype table

`../../data/meshwork_h5_subtypes.csv` maps each `.h5` file to its sensory subtype. The
root ID is read straight out of the meshwork file (`neuron.seg_id`) — no CloudVolume
lookup needed — resolved to a current root ID if proofreading has moved it, then joined to
the CAVE `sensory_neuron_label` table.

The `tag` values in that table carry trailing spaces (`'sst '`, `'mrgd '`), which are
stripped here and again in the figure notebooks.


In [ ]:
import glob
import os

import pandas as pd
from meshparty import meshwork

label = client.materialize.query_table("sensory_neuron_label")
root_to_tag = (label.groupby("pt_root_id")["tag"]
                    .agg(lambda v: sorted(set(map(str, v)))[0])
                    .to_dict())

rows = []
for f in sorted(glob.glob(str(OUT_DIR / "*.h5"))):
    n = meshwork.load_meshwork(f)
    rows.append({"coord": os.path.basename(f)[:-3],
                 "h5_seg_id": int(n.seg_id),
                 "n_axodendritic": len(n.anno.pre_syn.df),
                 "n_axoaxonic": len(n.anno.post_syn.df)})
df = pd.DataFrame(rows)

# resolve root IDs made stale by later proofreading, then attach the subtype
ids = np.asarray(df.h5_seg_id.values, dtype=np.uint64)
current = []
for sid, ok in zip(df.h5_seg_id, client.chunkedgraph.is_latest_roots(ids)):
    current.append(int(sid) if ok else
                   int(np.atleast_1d(client.chunkedgraph.suggest_latest_roots(int(sid)))[0]))
df["current_root_id"] = current
df["subtype"] = df.current_root_id.map(root_to_tag).str.strip()

df.to_csv("../../data/meshwork_h5_subtypes.csv", index=False)
print(df.subtype.value_counts().to_string())
